# MAE en profundidad: por qué predice la mediana y cuándo preferirla

En [`01_funcion_de_costo.ipynb`](01_funcion_de_costo.ipynb) viste que el MAE
(*Mean Absolute Error*) castiga cada residuo en proporción directa a su
tamaño, sin el "extra" que el cuadrado le da a los errores grandes. En
[`04_mse_en_profundidad.ipynb`](04_mse_en_profundidad.ipynb) viste, con un
ejemplo numérico, que minimizar MSE da la media de tus datos y minimizar MAE
da la mediana. Aquí construimos esa misma idea sobre una recta completa, con
dos ejemplos — uno simple con un error de captura, y uno con el mismo dataset
real de viviendas de California del notebook anterior — y cerramos con una
guía de cuándo preferir MAE sobre MSE o Huber.

In [1]:
import numpy as np
import polars as pl
import plotly.express as px
import plotly.graph_objects as go
from sklearn.datasets import fetch_california_housing
from sklearn.linear_model import LinearRegression, QuantileRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
from sklearn.model_selection import train_test_split

## 1. De un solo número a una recta

Recordemos la idea de `04_mse_en_profundidad.ipynb`: para un conjunto fijo de
valores, el número que minimiza el error cuadrático medio es la media, y el
que minimiza el error absoluto medio es la mediana. Al ajustar una recta
$\hat y = wx + b$, scikit-learn hace algo parecido, pero **para cada valor de
`x`**: con MSE busca, en cierto sentido, "la media de las `y` que
corresponden a ese `x`"; con MAE busca "la mediana de esas `y`". La
diferencia entre ambas rectas se nota exactamente donde hay outliers.

## 2. Ejemplo simple: un error de captura en el precio de una vivienda

Ocho registros de venta de viviendas, todos coherentes salvo uno: alguien
tecleó un cero de más en el precio de una vivienda de 65 m² (1500 en vez de
150, en miles de USD).

In [2]:
datos = pl.DataFrame({
    "tamano_m2": [40, 55, 65, 75, 90, 100, 110, 65],
    "precio_miles_usd": [100, 130, 150, 170, 200, 220, 240, 1500],
    "registro": ["normal"] * 7 + ["error de captura"],
})
datos

tamano_m2,precio_miles_usd,registro
i64,i64,str
40,100,"""normal"""
55,130,"""normal"""
65,150,"""normal"""
75,170,"""normal"""
90,200,"""normal"""
100,220,"""normal"""
110,240,"""normal"""
65,1500,"""error de captura"""


In [3]:
X = datos.select("tamano_m2").to_numpy()
y = datos["precio_miles_usd"].to_numpy()

modelo_mse = LinearRegression().fit(X, y)
modelo_mae = QuantileRegressor(quantile=0.5, alpha=0.0, solver="highs").fit(X, y)

m2_linea = np.linspace(35, 115, 100).reshape(-1, 1)
fig = px.scatter(
    datos, x="tamano_m2", y="precio_miles_usd", color="registro",
    color_discrete_map={"normal": "black", "error de captura": "#d62728"},
    title="La recta de MSE se deja arrastrar por el error de captura; la de MAE lo ignora casi por completo",
)
fig.add_trace(go.Scatter(x=m2_linea.ravel(), y=modelo_mse.predict(m2_linea), mode="lines", name="MSE (LinearRegression)"))
fig.add_trace(go.Scatter(x=m2_linea.ravel(), y=modelo_mae.predict(m2_linea), mode="lines", name="MAE (QuantileRegressor)"))
fig.update_layout(xaxis_title="Tamaño (m²)", yaxis_title="Precio (miles USD)")
fig.show()

print(f"w MSE: {modelo_mse.coef_[0]:.2f}   b MSE: {modelo_mse.intercept_:.2f}")
print(f"w MAE: {modelo_mae.coef_[0]:.2f}   b MAE: {modelo_mae.intercept_:.2f}")

w MSE: -1.46   b MSE: 448.37
w MAE: 2.00   b MAE: 20.00


El resultado es más drástico de lo que parece a simple vista: la recta de
MSE queda con **pendiente negativa** (`w = -1.46`) — literalmente concluye
que las viviendas más grandes valen *menos*, lo opuesto a la relación real.
Un solo dato mal capturado, en un dataset de apenas 8 registros, fue
suficiente para invertir el sentido de la relación completa. La recta de MAE,
en cambio, recupera exactamente la relación original de los datos limpios
(`w = 2.00`, `b = 20.00`): la mediana en cada punto de `x` ignora casi por
completo ese único valor absurdo.

Esto ilustra bien por qué MSE es peligroso con datasets pequeños: un solo
outlier tiene mucho más "peso relativo" sobre el total cuando hay pocos
puntos para diluirlo.

## 3. Por qué usamos `QuantileRegressor(quantile=0.5)` y no un "MAERegressor"

Scikit-learn no tiene una clase llamada así porque, en la práctica, minimizar
MAE es un caso particular de un problema más general: la **regresión
cuantílica**. `QuantileRegressor(quantile=0.5)` busca la recta donde la mitad
de los residuos quedan por encima y la mitad por debajo — la definición misma
de mediana, ahora condicionada a `x`.

Hay una razón técnica para elegir esta clase en particular. La curva de MAE,
$|r|$, tiene una esquina exactamente en $r=0$ — no es "suave" ahí, así que no
tiene una derivada bien definida en ese punto (a diferencia de $r^2$, que sí
la tiene en todas partes, como viste en
[`04_mse_en_profundidad.ipynb`](04_mse_en_profundidad.ipynb)). El descenso de
gradiente que programaste a mano en
[`01_funcion_de_costo.ipynb`](01_funcion_de_costo.ipynb) depende de poder
calcular esa derivada en cada punto; por eso, para minimizar MAE de forma
exacta, `QuantileRegressor` no usa descenso de gradiente, sino un **solver de
programación lineal** (`solver="highs"`), un método distinto pensado
específicamente para este tipo de "esquinas".

## 4. Ejemplo complejo: el mismo tope de California, visto con MAE

Volvamos al dataset de `fetch_california_housing` de
[`04_mse_en_profundidad.ipynb`](04_mse_en_profundidad.ipynb), con el mismo
tope artificial de 500 000 USD. Esta vez, en lugar de quitar manualmente las
viviendas topadas para "arreglar" el entrenamiento, entrenamos un modelo MAE
sobre **todos** los datos (topados incluidos) y lo comparamos contra los dos
modelos MSE del notebook anterior, evaluando los tres sobre el mismo conjunto
de prueba limpio (sin viviendas topadas).

In [4]:
california = fetch_california_housing()
datos_california = pl.DataFrame({
    "ingreso_medio": california.data[:, list(california.feature_names).index("MedInc")],
    "precio_medio": california.target,
})
tope = datos_california["precio_medio"].max()

# Misma semilla y misma partición que en 04_mse_en_profundidad.ipynb, para que la comparación sea justa.
X_train, X_test, y_train, y_test = train_test_split(
    datos_california.select("ingreso_medio").to_numpy(),
    datos_california["precio_medio"].to_numpy(),
    test_size=0.3, random_state=42,
)
entrenamiento_limpio = datos_california.filter(pl.col("precio_medio") < tope)
X_train_limpio, X_test_limpio, y_train_limpio, y_test_limpio = train_test_split(
    entrenamiento_limpio.select("ingreso_medio").to_numpy(),
    entrenamiento_limpio["precio_medio"].to_numpy(),
    test_size=0.3, random_state=42,
)

modelo_mse_con_tope = LinearRegression().fit(X_train, y_train)
modelo_mse_sin_tope = LinearRegression().fit(X_train_limpio, y_train_limpio)
modelo_mae_con_tope = QuantileRegressor(quantile=0.5, alpha=0.0, solver="highs").fit(X_train, y_train)

resultados_california = pl.DataFrame([
    {
        "modelo": nombre,
        "w (ingreso_medio)": modelo_comparado.coef_[0],
        "MAE en prueba limpia": mean_absolute_error(y_test_limpio, modelo_comparado.predict(X_test_limpio)),
        "RMSE en prueba limpia": root_mean_squared_error(y_test_limpio, modelo_comparado.predict(X_test_limpio)),
    }
    for nombre, modelo_comparado in [
        ("MSE, con tope incluido", modelo_mse_con_tope),
        ("MSE, sin viviendas topadas", modelo_mse_sin_tope),
        ("MAE, con tope incluido", modelo_mae_con_tope),
    ]
])
resultados_california

modelo,w (ingreso_medio),MAE en prueba limpia,RMSE en prueba limpia
str,f64,f64,f64
"""MSE, con tope incluido""",0.418193,0.585895,0.760902
"""MSE, sin viviendas topadas""",0.398381,0.574707,0.757828
"""MAE, con tope incluido""",0.439941,0.561829,0.766945


El resultado es más matizado que "MAE siempre gana": el modelo MAE obtiene el
mejor **MAE** en la prueba (0.562, el más bajo de los tres) — algo esperable,
porque fue entrenado exactamente para minimizar esa métrica. Pero su
**RMSE** (0.767) resulta el *peor* de los tres, incluso peor que el modelo
MSE entrenado con el tope incluido (0.761). Y su pendiente (`w = 0.440`) no
queda entre las otras dos: es la más alta de las tres.

Esto no contradice nada de lo que viste antes: simplemente confirma que MSE y
MAE apuntan a objetivos distintos (media vs. mediana), y cada métrica premia
al modelo que fue construido para optimizarla. Con un dataset grande donde el
tope afecta preferentemente a los vecindarios de mayor ingreso, la mediana
condicional en esa zona también puede verse desplazada por el tope, aunque de
forma distinta a como lo hace la media. La conclusión práctica, la misma que
en [`03_huber_loss_en_profundidad.ipynb`](03_huber_loss_en_profundidad.ipynb):
no existe una pérdida universalmente mejor — hay que decidir primero qué
resumen te importa (¿el error típico o el error total?) y qué métrica vas a
usar para juzgar el resultado, y solo después elegir la pérdida coherente con
esa decisión.

## 5. Guía corta: cuándo elegir MAE

- Úsalo cuando quieres que el error "típico" (la mediana) sea el resumen que
  te importa, y no quieres que unos pocos casos extremos lo dominen — como en
  el ejemplo de las 8 viviendas, donde un solo dato bastó para invertir el
  signo de la relación bajo MSE.
- Úsalo cuando el costo de un error crece de forma **proporcional** a su
  tamaño, no más que proporcional (a diferencia de MSE, donde un error del
  doble de tamaño pesa cuatro veces más).
- Úsalo cuando sospechas que tu variable objetivo tiene errores de captura,
  outliers no corregidos o una distribución con cola larga, y prefieres una
  recta que no se deje arrastrar por ellos.
- Recuerda evaluarlo con la métrica correcta: si eliges MAE como pérdida de
  entrenamiento pero luego comparas modelos con RMSE, puedes llevarte una
  sorpresa — cada pérdida favorece a la métrica que optimiza.

Evítalo (o usa MSE) cuando de verdad te interesan los errores grandes por
encima de los típicos, cuando necesitas la media (por ejemplo, para sumar
predicciones individuales y obtener un total), o cuando vas a optimizar con
descenso de gradiente estándar y quieres evitar la complicación de la esquina
en $r=0$.

## 6. Ideas clave

- Minimizar MAE, punto por punto, equivale a buscar la mediana condicional de
  `y` dado `x` — por eso scikit-learn lo implementa como un caso de regresión
  cuantílica (`QuantileRegressor(quantile=0.5)`), no como una clase aparte.
- MAE es robusto porque la mediana ignora "cuánto" se aleja un valor extremo,
  solo le importa que quede de un lado o del otro — a diferencia de la media,
  que sí carga con la magnitud completa del extremo.
- La curva de MAE tiene una esquina en $r=0$ (no es diferenciable ahí), así
  que su optimización exacta usa un solver distinto al descenso de gradiente
  que programaste a mano para MSE.
- Ninguna pérdida es universalmente mejor: MAE minimiza mejor el error típico,
  pero puede quedar peor que MSE si tu criterio final de evaluación es RMSE.

**Ejercicio:** en el ejemplo simple de la sección 2, cambia el precio del
registro con error de captura de 1500 a un valor menos extremo (por ejemplo,
300) y vuelve a comparar `w` de ambos modelos. Antes de ejecutar, predice:
¿la pendiente de MSE seguirá siendo negativa, o volverá a ser positiva? ¿A
partir de qué tan grande tiene que ser el error de captura para que MSE
invierta el signo de la relación?